In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from figures.generate_data import load_image_point_cloud
from diffusion_geometry import DiffusionGeometry


Create some data

In [ ]:
data1 = np.random.uniform(-1, 1, (2000, 2))

# sample data1 from a grid:
x, y = np.meshgrid(np.linspace(-1, 1, 40), np.linspace(-1, 1, 40))
data1 = np.vstack([x.ravel(), y.ravel()]).T
data1 += 0.015 * np.random.randn(*data1.shape)

plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
x, y = data1[:,0], data1[:,1]
f = dg1.function(np.sin(5*x) + np.cos(5*y))
Hf = f.hessian()

scatter_size = 4

f11 = plot_scatter_2d(data1, f.to_ambient(), size=scatter_size)
f11.show()
f12 = plot_quiver_2d(data1, f.grad().to_ambient(), scale=0.015, line_width=1)
f12.show()
f13 = plot_ellipsoids(data1, Hf.to_ambient(), scale=0.01)
f13.show()

dx = dg1.function(x).grad()
dy = dg1.function(y).grad()
Hxx = Hf(dx, dx)
Hxy = Hf(dx, dy)
Hyy = Hf(dy, dy)

amax = max(np.max(np.abs(Hxx)), np.max(np.abs(Hyy)), np.max(np.abs(Hxy)))
f21 = plot_scatter_2d(data1, Hxx, size=scatter_size, amax=amax)
f21.show()
f22 = plot_scatter_2d(data1, Hyy, size=scatter_size, amax=amax)
f22.show()
f23 = plot_scatter_2d(data1, Hxy, size=scatter_size, amax=amax)
f23.show()

In [ ]:
from generate_data import gen_3d_data
data2, _ = gen_3d_data(kind = "swiss_roll", n = 3000, noise = 0.0)
dg2 = DiffusionGeometry.from_point_cloud(data2)

camera = dict(eye=dict(x=-0.2, y=-1.7, z=0.5), center=dict(x=0.04, y=0, z=-0.08))
plot_scatter_3d(data2, color = 'black', camera=camera).show()

In [ ]:
f = dg2.function(data2[:,2])
Hf = f.hessian()

scatter_size = 2.5

f31 = plot_scatter_3d(data2, f.to_ambient(), size=scatter_size, camera=camera)
f31.show()
f32 = plot_quiver_3d(data2, f.grad().to_ambient(), scale=0.8, line_width=1, arrow_scale=1.5)
f32.update_layout(scene_camera=camera)
f32.show()
f33 = plot_ellipsoids(data2, Hf.to_ambient(), scale=3, n_theta=8, n_phi=8)
f33.update_layout(scene_camera=camera)
# f33.show()

x, y, z = data2[:,0], data2[:,1], data2[:,2]
dx = dg2.function(x).grad()
dy = dg2.function(y).grad()
dz = dg2.function(z).grad()
Hxx = Hf(dx, dx)
Hxy = Hf(dx, dy)
Hyy = Hf(dy, dy)
Hyz = Hf(dy, dz)
Hzz = Hf(dz, dz)
Hxz = Hf(dx, dz)

amax = max(np.max(np.abs(Hxx)), np.max(np.abs(Hyy)), np.max(np.abs(Hxy)), np.max(np.abs(Hxz)), np.max(np.abs(Hyz)), np.max(np.abs(Hzz)))
f41 = plot_scatter_3d(data2, Hxx, size=scatter_size, amax=amax, camera=camera)
f41.show()
f42 = plot_scatter_3d(data2, Hzz, size=scatter_size, amax=amax, camera=camera)
# f42.show()
f43 = plot_scatter_3d(data2, Hxz, size=scatter_size, amax=amax, camera=camera)
# f43.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=4,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.04,
    # row_heights=[0.24, 0.24, 0.26, 0.26],
)

# --- Add traces row by row ---
fig.add_traces(list(f11.data), rows=[1] * len(f11.data), cols=[1] * len(f11.data))
fig.add_traces(list(f12.data), rows=[1] * len(f12.data), cols=[2] * len(f12.data))
fig.add_traces(list(f13.data), rows=[1] * len(f13.data), cols=[3] * len(f13.data))
fig.add_traces(list(f21.data), rows=[2] * len(f21.data), cols=[1] * len(f21.data))
fig.add_traces(list(f22.data), rows=[2] * len(f22.data), cols=[2] * len(f22.data))
fig.add_traces(list(f23.data), rows=[2] * len(f23.data), cols=[3] * len(f23.data))
fig.add_traces(list(f31.data), rows=[3] * len(f31.data), cols=[1] * len(f31.data))
fig.add_traces(list(f32.data), rows=[3] * len(f32.data), cols=[2] * len(f32.data))
fig.add_traces(list(f33.data), rows=[3] * len(f33.data), cols=[3] * len(f33.data))
fig.add_traces(list(f41.data), rows=[4] * len(f41.data), cols=[1] * len(f41.data))
fig.add_traces(list(f42.data), rows=[4] * len(f42.data), cols=[2] * len(f42.data))
fig.add_traces(list(f43.data), rows=[4] * len(f43.data), cols=[3] * len(f43.data))

# --- Synchronise 2D axis ranges ---
all_2d = [f11, f12, f13, f21, f22, f23]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f31, f32, f33, f41, f42, f43]
# all_3d = [f31, f41]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 7):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=1400,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1 or row == 3:
        if col == 1:
            return rf"$f$"
        elif col == 2:
            return rf"$\nabla f$"
        elif col == 3:
            return rf"$H(f)$"
    if row == 2:
        if col == 1:
            return rf"$H(f)(\nabla x, \nabla x)$"
        elif col == 2:
            return rf"$H(f)(\nabla y, \nabla y)$"
        elif col == 3:
            return rf"$H(f)(\nabla x, \nabla y)$"
    if row == 4:
        if col == 1:
            return rf"$H(f)(\nabla x, \nabla x)$"
        if col == 2:
            return rf"$H(f)(\nabla z, \nabla z)$"
        elif col == 3:
            return rf"$H(f)(\nabla x, \nabla z)$"
    else:
        return ""

print("top 2 rows")
overpic_labels(fig, labels, 
               stretch_x=1.0, 
            #    offset_x=-2,
               stretch_y=1.0,
               offset_y=13.)

print("bottom 2 rows")
overpic_labels(fig, labels, 
               stretch_x=1.0, 
            #    offset_x=-2,
               stretch_y=1.0,
               offset_y=12.5)



In [ ]:
from generate_data import gen_3d_data

n = 4000
data3 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = n, noise = 0.0, seed = 0)[0]
data3 = data3[:, [1, 2, 0]]

dg3 = DiffusionGeometry.from_point_cloud(data3)
f = dg3.function(data3[:,2])
gradf = f.grad()
Hf = f.hessian()

crit_points = np.exp(-gradf.pointwise_norm()/0.3)

scale_function = crit_points / np.max(crit_points)
scale_function = 2 * np.clip(scale_function, 0, 0.5)

scatter_size = 1
camera3 = dict(eye=dict(x=2, y=1.5, z=0.3),
                center=dict(x=0.12, y=0, z=0.02),
                up=dict(x=0, y=0, z=1))
camera3["eye"] = {k: 0.8 * v for k, v in camera3["eye"].items()}
plot_scatter_3d(data3, color='black', size = scatter_size, camera=camera3).update_layout(width=250, height=300).show()

In [ ]:
f51 = plot_scatter_3d(data3, f.to_ambient(), size=scatter_size, camera=camera3)
f51.show()

f52 = plot_quiver_3d(data3, gradf.to_ambient(), scale=0.2, line_width=1, arrow_scale=1)
f52.update_layout(scene_camera=camera3)
f52.show()

f53 = plot_scatter_3d(data3, crit_points, size=2 * scatter_size * scale_function, camera=camera3)
f53.show()

f54 = plot_ellipsoids(data3, Hf.to_ambient(), scale=0.2, n_theta=8, n_phi=8)
f54.update_layout(scene_camera=camera3)
f54.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=1,
    cols=4,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.0,
)

# --- Add traces row by row ---
fig.add_traces(list(f51.data), rows=[1] * len(f51.data), cols=[1] * len(f51.data))
fig.add_traces(list(f52.data), rows=[1] * len(f52.data), cols=[2] * len(f52.data))
fig.add_traces(list(f53.data), rows=[1] * len(f53.data), cols=[3] * len(f53.data))
fig.add_traces(list(f54.data), rows=[1] * len(f54.data), cols=[4] * len(f54.data))


# --- Synchronise 3D ranges ---
all_3d = [f51, f52, f53, f54]
x3d, y3d, z3d = [], [], []
for figure in all_3d:
    for t in figure.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 5):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera3,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=300,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/new_fig.pdf", scale=1)


In [ ]:
camera4 = dict(eye=dict(x=0.6, y=0.8, z=1.1),
                center=dict(x=0.08, y=0, z=0.15),
                up=dict(x=0, y=0, z=1))

f61 = plot_quiver_3d(data3, gradf.to_ambient(), scale=0.3, line_width=1, arrow_scale=0.3, opacity=scale_function**1.5)
f61.update_layout(scene_camera=camera4)
f61.update_layout(
    width=500,
    height=500,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)
f61.show()

In [ ]:
x, y, z = data3[:,0], data3[:,1], data3[:,2]
X = dg3.function(x).grad()
Y = dg3.function(y).grad()

evals, evecs = dg3.laplacian(1).spectrum()
X, Y = evecs[0].sharp(), evecs[1].sharp()

Hxx = Hf(X, X)
Hxy = Hf(X, Y)
Hyy = Hf(Y, Y)
hess_matrix = np.stack((Hxx,Hxy,Hxy,Hyy), axis=1).reshape(-1,2,2)

Gxx = dg3.g(X, X)
Gxy = dg3.g(X, Y)
Gyy = dg3.g(Y, Y)
gram_matrix = np.stack((Gxx,Gxy,Gxy,Gyy), axis=1).reshape(-1,2,2)

import scipy as sp

vals_list = []
vecs_list = []
for i in range(dg3.n):
    vals, vecs = sp.linalg.eigh(hess_matrix[i], gram_matrix[i])
    vals_list.append(vals)
    vecs_list.append(vecs)
vals = np.array(vals_list)  # (n,2)
vecs = np.array(vecs_list)  # (n,2,2)

index = (vals < 0).sum(axis=1)

vec_1 = vecs[:, :, 0]
vec_2 = vecs[:, :, 1]

ambient_basis = np.stack((X.to_ambient(), Y.to_ambient()), axis=1)  # (n,2,3)
vecs_ambient = np.einsum('pia,pij->paj', ambient_basis, vecs)  # (n,3,2)

In [ ]:
f62 = plot_hessian_eig_lines(data3, vecs_ambient, vals, scale=0.1)
f62.update_layout(scene_camera=camera4)
f62.show()

f63 = plot_scatter_3d(data3, (1 - index)*scale_function, size=4*scatter_size*scale_function, camera=camera4,)
f63.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# --- Define subplot grid ---
specs = [[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]]
fig = make_subplots(
    rows=1,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.0,
)

# --- Add traces row by row ---
fig.add_traces(list(f61.data), rows=[1] * len(f61.data), cols=[1] * len(f61.data))
fig.add_traces(list(f62.data), rows=[1] * len(f62.data), cols=[2] * len(f62.data))
fig.add_traces(list(f63.data), rows=[1] * len(f63.data), cols=[3] * len(f63.data))

# --- Synchronise 3D ranges ---
all_3d = [f61, f62, f63]
x3d, y3d, z3d = [], [], []
for figure in all_3d:
    for t in figure.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]

camera4_zoomout = dict(
    eye={k: 1.15 * v for k, v in camera4["eye"].items()},
    center=camera4["center"],
    up=camera4["up"],
)

for i in range(1, 4):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        key: dict(
            camera=camera4_zoomout,
            xaxis=dict(range=xrange3),
            yaxis=dict(range=yrange3),
            zaxis=dict(range=zrange3),
            aspectmode="data",
        )
    })

# --- Compute limits from vals array ---
amax = float(np.nanmax(np.abs(vals)))
vmin, vmax = -amax, amax

# --- Continuous colorbar (middle plot) ---
fig.add_trace(
    go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode="markers",
        marker=dict(
            colorscale=[[0, "#166dde"], [0.5, "lightgray"], [1, "#e32636"]],
            cmin=vmin, cmax=vmax,
            showscale=True,
            colorbar=dict(
                title="",
                tickvals=[-1, 0, 1],
                ticktext=["−1", "0", "1"],
                ticks="outside",
                len=0.25,          # shorter
                thickness=10,      # thinner
                outlinewidth=0,
                tickfont=dict(size=10),
                x=0.5,             # centered above middle subplot
                y=0.98,            # closer to plot
                xanchor="center",
                yanchor="bottom",
                orientation="h",
                bgcolor="rgba(0,0,0,0)",
            ),
        ),
        hoverinfo="none",
    )
)

# --- Discrete categorical colorbar (right plot) ---
fig.add_trace(
    go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode="markers",
        marker=dict(
            color=[0, 1, 2],
            cmin=-0.5, cmax=2.5,
            colorscale=[
                [0.0, "#e32636"], [0.3333, "#e32636"],
                [0.3334, "lightgray"], [0.6666, "lightgray"],
                [0.6667, "#166dde"], [1.0, "#166dde"],
            ],
            showscale=True,
            colorbar=dict(
                title="",
                tickvals=[0, 1, 2],
                ticktext=["0", "1", "2"],
                ticks="outside",
                len=0.25,
                thickness=10,
                outlinewidth=0,
                tickfont=dict(size=10),
                x=0.83,             # aligned over right subplot
                y=0.98,             # closer to plot
                xanchor="center",
                yanchor="bottom",
                orientation="h",
                bgcolor="rgba(0,0,0,0)",
            ),
        ),
        hoverinfo="none",
    )
)

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=360,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.write_image("figs/new_fig.pdf", scale=2)
fig.show()


In [ ]:
def labels(row, col):
    if row == 1:
        if col == 1:
            return rf"$\nabla f$ near critical points"
        elif col == 2:
            return rf"eigenvectors of $H(f)$"
        elif col == 3:
            return rf"Morse indices near critical points"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=1.0, 
            #    offset_x=-2,
               stretch_y=1.0,
               offset_y=19.5)
